In [ ]:
# Outliers

print("Lap time outlier check:")
print(f"  Minimum lap time: {lap_times['milliseconds'].min() / 1000:.1f} seconds")
print(f"  Maximum lap time: {lap_times['milliseconds'].max() / 1000:.1f} seconds")
print(f"  Laps over 5 minutes (300s): {(lap_times['milliseconds'] > 300_000).sum():,}")
print()

# Show the largest outliers to understand the cause
worst = lap_times.nlargest(3, 'milliseconds')[['raceId', 'driverId', 'lap', 'milliseconds']]
worst['seconds'] = worst['milliseconds'] / 1000
print("Top 3 slowest recorded laps:")
print(worst.to_string(index=False))
print()

# remove laps that are outside 80%–150% of the race median - arbitrary
# removes SC laps and extreme outliers
# all laps less than 80% of median and greater tha 150% of median
print("Removing laps that are outside 80%–150% of the race median")
print()
race_median = lap_times.groupby('raceId')['milliseconds'].median().rename('race_med')
lap_times = lap_times.drop(columns=['race_med'], errors='ignore').join(race_median, on='raceId')
lap_times_clean = lap_times[
    (lap_times['milliseconds'] > lap_times['race_med'] * 0.80) &
    (lap_times['milliseconds'] < lap_times['race_med'] * 1.50)
].copy()

removed = len(lap_times) - len(lap_times_clean)
print(f"Laps removed as outliers: {removed:,} ({removed/len(lap_times)*100:.2f}%)")
print(f"Laps kept for analysis:   {len(lap_times_clean):,}")
